## Orientation Flipping Experiment 

I wanted to see whether randomly flipping the images (horizontally and vertically) so that the model sees more orientations helps the model classify the held out images better. 
I want to use seeds 123 and 124 to compare no flips with random horizontal and vertical flips. 
I will use the same model settings for both with
- padding=1
- epochs=10
- batch size=64
- Learning rate = 0.001
- filters= 16->32
- optimiser = Adam

### What the code does

Each time an image is loaded it has a 50% chance of flipping left to right and a 50% chance of flipping top to bottom (these are independent). So it can flip both, flip one or flip neither. 

I'll also rerun the no flips comparison in this notebook so the timings come from the same session. 


In [4]:
from pathlib import Path
from time import perf_counter

import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Find the shared training folder from the repo root or the Alice folder.
train_path = Path("data/train")
if not train_path.is_dir():
    train_path = Path("../../data/train")

plain_data = ImageFolder(train_path, transform=transforms.ToTensor())

flipped_data = ImageFolder(
    train_path,
    transform=transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
    ]),
)

# Recreate the same stratified split used in CNN.ipynb.
train_indices, val_indices = train_test_split(
    range(len(plain_data)),
    test_size=0.20,
    random_state=7,
    stratify=plain_data.targets,
)

training_sets = {
    "No flips": Subset(plain_data, train_indices),
    "Random flips": Subset(flipped_data, train_indices),
}

validation = Subset(plain_data, val_indices)
val_loader = DataLoader(validation, batch_size=32, shuffle=False)

print(f"Training images per trial: {len(train_indices):,}")
print(f"Validation images per trial: {len(validation):,}")

Training images per trial: 17,280
Validation images per trial: 4,320


In [5]:
seeds = (123, 124)
num_epochs = 10


def build_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 16 * 16, 10),
    )


def run_flip_trial(training_data, seed):
    torch.manual_seed(seed)
    model = build_cnn()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_function = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        training_data,
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )

    history = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        loss_sum = 0.0
        start_time = perf_counter()

        for images, labels in train_loader:
            optimizer.zero_grad()
            scores = model(images)
            loss = loss_function(scores, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * labels.size(0)

        training_seconds = perf_counter() - start_time
        training_loss = loss_sum / len(training_data)

        model.eval()
        correct = 0
        with torch.inference_mode():
            for images, labels in val_loader:
                predictions = model(images).argmax(dim=1)
                correct += (predictions == labels).sum().item()

        accuracy = correct / len(validation)
        history.append({
            "validation_accuracy": accuracy,
            "training_seconds": training_seconds,
        })

        print(
            f"Epoch {epoch:2d}/{num_epochs} | Loss {training_loss:.3f} | "
            f"Validation accuracy {accuracy:.2%} | "
            f"Training time {training_seconds:.1f}s",
            flush=True,
        )

    return history


flip_results = {}

for seed in seeds:
    for name, training_data in training_sets.items():
        print(f"\n{name} | Seed {seed}", flush=True)
        flip_results[(name, seed)] = run_flip_trial(training_data, seed)

print("\nMean validation accuracy across both seeds")
print(f"{'Epoch':>5} | {'No flips':>10} | {'Random flips':>12}")

for index in range(num_epochs):
    means = [
        sum(flip_results[(name, seed)][index]["validation_accuracy"] for seed in seeds)
        / len(seeds)
        for name in training_sets
    ]
    print(f"{index + 1:>5} | {means[0]:>10.2%} | {means[1]:>12.2%}")

print("\nEpoch-10 accuracy and total training time, excluding validation")
for (name, seed), history in flip_results.items():
    total_time = sum(row["training_seconds"] for row in history)
    print(
        f"{name:>12} | Seed {seed} | "
        f"{history[-1]['validation_accuracy']:.2%} | {total_time:.1f}s"
    )


No flips | Seed 123
Epoch  1/10 | Loss 1.384 | Validation accuracy 63.54% | Training time 20.9s
Epoch  2/10 | Loss 0.934 | Validation accuracy 67.89% | Training time 20.3s
Epoch  3/10 | Loss 0.793 | Validation accuracy 73.75% | Training time 20.3s
Epoch  4/10 | Loss 0.696 | Validation accuracy 77.89% | Training time 21.2s
Epoch  5/10 | Loss 0.634 | Validation accuracy 79.35% | Training time 20.2s
Epoch  6/10 | Loss 0.603 | Validation accuracy 78.84% | Training time 20.4s
Epoch  7/10 | Loss 0.556 | Validation accuracy 79.86% | Training time 20.4s
Epoch  8/10 | Loss 0.528 | Validation accuracy 75.65% | Training time 20.3s
Epoch  9/10 | Loss 0.518 | Validation accuracy 77.82% | Training time 18.5s
Epoch 10/10 | Loss 0.479 | Validation accuracy 82.48% | Training time 15.5s

Random flips | Seed 123
Epoch  1/10 | Loss 1.380 | Validation accuracy 64.65% | Training time 16.0s
Epoch  2/10 | Loss 0.928 | Validation accuracy 70.93% | Training time 16.1s
Epoch  3/10 | Loss 0.800 | Validation accu

## Results - Orientation 

**Seed=123**
No flips had the highest accuracy at epoch 10 with 82.48% compared with flips with 82.06%. 

**Seed=124**
The highest accuracy for no flips at 80.37% was actually lower than that for flips at 8-.49% both achieved at epoch 9. 

However for both when average the mean validation accuracy at epoch 20 was 81.12% for no flips and 80.80% for flips and flips was 5.8 seconds slower to run. At epoch 9 it also had a lower mean validation accuracy so in conclusion i think keeping the model with no flips is the right option. 